# SpaceX Falcon 9 — Exploratory Data Analysis with SQL

We load the launch record table into an in-memory SQLite database and answer a
series of analysis questions with plain SQL, exactly as required by the
capstone's EDA-with-SQL lab.

In [1]:
import pandas as pd
import sqlite3
import re

BASE = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork"
df = pd.read_csv(f"{BASE}/labs/module_2/data/Spacex.csv")
df.columns = [re.sub(r"[^0-9a-zA-Z_]+", "_", col).strip("_") for col in df.columns]

conn = sqlite3.connect(":memory:")
df.to_sql("SPACEXTBL", conn, index=False, if_exists="replace")
df.head()

,Date,Time_UTC,Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG,Orbit,Customer,Mission_Outcome,Landing_Outcome
0,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
3,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


### Unique launch sites

In [2]:
pd.read_sql("SELECT DISTINCT Launch_Site FROM SPACEXTBL;", conn)

,Launch_Site
0,CCAFS LC-40
1,VAFB SLC-4E
2,KSC LC-39A
3,CCAFS SLC-40


### 5 records where the launch site begins with `CCA`

In [3]:
pd.read_sql("SELECT * FROM SPACEXTBL WHERE Launch_Site LIKE 'CCA%' LIMIT 5;", conn)

,Date,Time_UTC,Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG,Orbit,Customer,Mission_Outcome,Landing_Outcome
0,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
3,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


### Total payload mass carried by boosters launched for NASA (CRS)

In [4]:
pd.read_sql("SELECT SUM(PAYLOAD_MASS__KG) AS Total_Payload_Mass FROM SPACEXTBL WHERE Customer LIKE '%NASA (CRS)%';", conn)

,Total_Payload_Mass
0,48213


### Average payload mass carried by booster version F9 v1.1

In [5]:
pd.read_sql("SELECT AVG(PAYLOAD_MASS__KG) AS Avg_Payload_Mass FROM SPACEXTBL WHERE Booster_Version LIKE 'F9 v1.1%';", conn)

,Avg_Payload_Mass
0,2534.666667


### Date of the first successful ground-pad landing

In [6]:
pd.read_sql("SELECT MIN(Date) AS First_Success_Ground_Landing FROM SPACEXTBL WHERE Landing_Outcome = 'Success (ground pad)';", conn)

,First_Success_Ground_Landing
0,2015-12-22


### Boosters that landed successfully on a drone ship with 4000–6000 kg payload

In [7]:
pd.read_sql("SELECT Booster_Version FROM SPACEXTBL WHERE Landing_Outcome = 'Success (drone ship)' AND PAYLOAD_MASS__KG > 4000 AND PAYLOAD_MASS__KG < 6000;", conn)

,Booster_Version
0,F9 FT B1022
1,F9 FT B1026
2,F9 FT B1021.2
3,F9 FT B1031.2


### Total successful vs. failed mission outcomes

In [8]:
pd.read_sql("SELECT Mission_Outcome, COUNT(*) AS Count FROM SPACEXTBL GROUP BY Mission_Outcome;", conn)

,Mission_Outcome,Count
0,Failure (in flight),1
1,Success,98
2,Success,1
3,Success (payload status unclear),1


### Booster(s) that carried the maximum payload mass

In [9]:
pd.read_sql('''
SELECT Booster_Version FROM SPACEXTBL
WHERE PAYLOAD_MASS__KG = (SELECT MAX(PAYLOAD_MASS__KG) FROM SPACEXTBL);
''', conn)

,Booster_Version
0,F9 B5 B1048.4
1,F9 B5 B1049.4
2,F9 B5 B1051.3
3,F9 B5 B1056.4
4,F9 B5 B1048.5
5,F9 B5 B1051.4
6,F9 B5 B1049.5
7,F9 B5 B1060.2
8,F9 B5 B1058.3
9,F9 B5 B1051.6


### 2015 drone-ship landing failures, by month/booster/site

In [10]:
pd.read_sql('''
SELECT substr(Date,6,2) AS Month, Landing_Outcome, Booster_Version, Launch_Site
FROM SPACEXTBL
WHERE Landing_Outcome = 'Failure (drone ship)' AND substr(Date,1,4) = '2015';
''', conn)

,Month,Landing_Outcome,Booster_Version,Launch_Site
0,01,Failure (drone ship),F9 v1.1 B1012,CCAFS LC-40
1,04,Failure (drone ship),F9 v1.1 B1015,CCAFS LC-40


### Rank landing outcome counts between 2010-06-04 and 2017-03-20

In [11]:
pd.read_sql('''
SELECT Landing_Outcome, COUNT(*) AS Outcome_Count
FROM SPACEXTBL
WHERE Date BETWEEN '2010-06-04' AND '2017-03-20'
GROUP BY Landing_Outcome
ORDER BY Outcome_Count DESC;
''', conn)

,Landing_Outcome,Outcome_Count
0,No attempt,10
1,Success (drone ship),5
2,Failure (drone ship),5
3,Success (ground pad),3
4,Controlled (ocean),3
5,Uncontrolled (ocean),2
6,Failure (parachute),2
7,Precluded (drone ship),1
